Ways to create a delta table

Using delta method

In [0]:
%sql
create or replace table retail_products_sql(
    product_id int,
    product_name string,
    price int
)
using delta;

In [0]:
%sql
insert into retail_products_sql values
(1,'Laptop',50000),
(2,'Mouse',800),
(3,'Keyboard',1500);

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
select * from retail_products_sql;

product_id,product_name,price
1,Laptop,50000
2,Mouse,800
3,Keyboard,1500


Using format Method

In [0]:
data = [
    (4, "Monitor", 12000),
    (5, "Printer", 18000),
    (6, "Webcam", 2500)
]
 
df = spark.createDataFrame(
    data,
    ["product_id", "product_name", "price"]
)
 
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail_products_pyspark")
 
display(spark.table("retail_products_pyspark"))

product_id,product_name,price
4,Monitor,12000
5,Printer,18000
6,Webcam,2500


Create Table As Select(CTAS) Method 

In [0]:
%sql
CREATE OR REPLACE TABLE retail_products_ctas
USING DELTA
AS
SELECT *
FROM VALUES
    (7, 'Tablet', 25000),
    (8, 'Speaker', 5000),
    (9, 'Router', 3500)
AS products(product_id, product_name, price);

num_affected_rows,num_inserted_rows


To view 

In [0]:
%sql
SELECT * FROM retail_products_ctas;

product_id,product_name,price
7,Tablet,25000
8,Speaker,5000
9,Router,3500


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW product_updates AS
SELECT *
FROM VALUES
    (2, 'Wireless Mouse', 950),
    (10, 'USB Hub', 1200)
AS updates(product_id, product_name, price);


In [0]:

%sql
MERGE INTO retail_products_sql AS target
USING product_updates AS source
ON target.product_id = source.product_id

WHEN MATCHED THEN
    UPDATE SET
        target.product_name = source.product_name,
        target.price = source.price

WHEN NOT MATCHED THEN
    INSERT (product_id, product_name, price)
    VALUES (source.product_id, source.product_name, source.price);
 

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,0,1


In [0]:

%sql
SELECT * FROM retail_products_sql
ORDER BY product_id;

product_id,product_name,price
1,Laptop,50000
2,Wireless Mouse,950
3,Keyboard,1500
10,USB Hub,1200


In [0]:
%sql
describe detail retail_products_sql;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,402ac34b-ca98-4910-9d88-c1cd18f06ad8,pattu_databricks.default.retail_products_sql,null,abfss://unity-catalog-storage@dbstoragea6fcql23b2xjm.dfs.core.windows.net/7405613477004258/__unitystorage/catalogs/43771f81-6983-4372-be62-f9080e253ef5/tables/ef7c1817-c899-457f-9144-8e60796c3bb2,2026-11-05T07:20:36.484Z,2026-11-05T08:50:33Z,List(),List(),1,2190,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true, delta.enableRowTracking -> true, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-c29c2950-e8a6-445b-9c1d-0cd8848da916, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-dad2dfb0-685f-4188-a35e-83ba286c0d0e)",3,7,"List(appendOnly, deletionVectors, domainMetadata, invariants, rowTracking)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
describe history retail_products_sql;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-11-05T08:50:33Z,144579690930574,azuser7414_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3401750592353526),null,1103-072026-gpfcj4bf,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 5272, p25FileSize -> 2190, numDeletionVectorsRemoved -> 1, minFileSize -> 2190, numAddedFiles -> 1, maxFileSize -> 2190, p75FileSize -> 2190, p50FileSize -> 2190, numAddedBytes -> 2190)",null,Databricks-Runtime/18.x-photon-scala2.13
2,2026-11-05T08:50:29Z,144579690930574,azuser7414_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(product_id#1009 = product_id#1000)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3401750592353526),null,1103-072026-gpfcj4bf,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 3981, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 7558, materializeSourceTimeMs -> 243, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 4164, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 3094)",null,Databricks-Runtime/18.x-photon-scala2.13
1,2026-11-05T07:20:47Z,144579690930574,azuser7414_mml.local@karthikirisoutlook.onmicrosoft.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(3401750592353526),null,1103-072026-gpfcj4bf,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 1291)",null,Databricks-Runtime/18.x-photon-scala2.13
0,2026-11-05T07:20:37Z,144579690930574,azuser7414_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-c29c2950-e8a6-445b-9c1d-0cd8848da916"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-dad2dfb0-685f-4188-a35e-83ba286c0d0e""}, statsOnLoad -> false)",null,List(3401750592353526),null,1103-072026-gpfcj4bf,null,WriteSerializable,true,Map(),null,Databricks-Runtime/18.x-photon-scala2.13


In [0]:
%sql
select * from retail_products_sql version as of 1;

product_id,product_name,price
1,Laptop,50000
2,Mouse,800
3,Keyboard,1500


In [0]:
%sql
optimize retail_products_sql;

path,metrics
abfss://unity-catalog-storage@dbstoragea6fcql23b2xjm.dfs.core.windows.net/7405613477004258/__unitystorage/catalogs/43771f81-6983-4372-be62-f9080e253ef5/tables/ef7c1817-c899-457f-9144-8e60796c3bb2,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1793869643358, 1793869644180, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, null, null)"


Zorder --> used to group relevant things together which improves performance

In [0]:
%sql
optimize retail_products_sql
zorder by (product_id);

path,metrics
abfss://unity-catalog-storage@dbstoragea6fcql23b2xjm.dfs.core.windows.net/7405613477004258/__unitystorage/catalogs/43771f81-6983-4372-be62-f9080e253ef5/tables/ef7c1817-c899-457f-9144-8e60796c3bb2,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 2190), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1793870026922, 1793870027598, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, null, null)"
